In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import sqlite3
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

class CoronaDetectionSystem:
    def __init__(self, root):
        self.root = root
        self.root.title("Corona Detection System")
        self.root.geometry("1200x700")
        self.root.configure(bg='#f0f8ff')
        
        # Initialize database
        self.init_database()
        
        # Create main frame
        self.main_frame = ttk.Frame(root, padding="10")
        self.main_frame.pack(fill=tk.BOTH, expand=True)
        
        # Create notebook for tabs
        self.notebook = ttk.Notebook(self.main_frame)
        self.notebook.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Create tabs
        self.create_patient_tab()
        self.create_test_tab()
        self.create_results_tab()
        self.create_analytics_tab()
        
        # Load initial data
        self.load_patients()
        self.load_tests()
        
    def init_database(self):
        """Initialize SQLite database"""
        self.conn = sqlite3.connect('corona_detection.db')
        self.cursor = self.conn.cursor()
        
        # Create patients table
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS patients (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                age INTEGER NOT NULL,
                gender TEXT NOT NULL,
                phone TEXT,
                address TEXT,
                registration_date TEXT
            )
        ''')
        
        # Create tests table
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS tests (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                patient_id INTEGER,
                test_date TEXT,
                temperature REAL,
                cough INTEGER,
                fever INTEGER,
                breathing_difficulty INTEGER,
                loss_of_taste_smell INTEGER,
                fatigue INTEGER,
                chest_pain INTEGER,
                travel_history INTEGER,
                contact_with_infected INTEGER,
                test_result TEXT,
                severity TEXT,
                FOREIGN KEY (patient_id) REFERENCES patients (id)
            )
        ''')
        
        self.conn.commit()
    
    def create_patient_tab(self):
        """Create patient registration tab"""
        self.patient_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.patient_tab, text="Patient Registration")
        
        # Patient form
        ttk.Label(self.patient_tab, text="Patient Registration", 
                 font=('Arial', 16, 'bold')).grid(row=0, column=0, columnspan=2, pady=10)
        
        # Form fields
        fields = [
            ("Name", "name_entry"),
            ("Age", "age_entry"),
            ("Gender", "gender_combo"),
            ("Phone", "phone_entry"),
            ("Address", "address_entry")
        ]
        
        self.patient_entries = {}
        for i, (label, key) in enumerate(fields):
            ttk.Label(self.patient_tab, text=label+":").grid(row=i+1, column=0, sticky='w', padx=5, pady=5)
            if key == "gender_combo":
                entry = ttk.Combobox(self.patient_tab, values=["Male", "Female", "Other"], state="readonly")
            else:
                entry = ttk.Entry(self.patient_tab, width=30)
            entry.grid(row=i+1, column=1, padx=5, pady=5)
            self.patient_entries[key] = entry
        
        # Buttons
        button_frame = ttk.Frame(self.patient_tab)
        button_frame.grid(row=6, column=0, columnspan=2, pady=10)
        
        ttk.Button(button_frame, text="Register Patient", 
                  command=self.register_patient).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Clear", 
                  command=self.clear_patient_form).pack(side=tk.LEFT, padx=5)
        
        # Patient list
        ttk.Label(self.patient_tab, text="Registered Patients", 
                 font=('Arial', 12, 'bold')).grid(row=7, column=0, columnspan=2, pady=10)
        
        self.patient_tree = ttk.Treeview(self.patient_tab, columns=("ID", "Name", "Age", "Gender", "Phone"), show="headings")
        for col in ["ID", "Name", "Age", "Gender", "Phone"]:
            self.patient_tree.heading(col, text=col)
            self.patient_tree.column(col, width=100)
        self.patient_tree.grid(row=8, column=0, columnspan=2, padx=5, pady=5)
    
    def create_test_tab(self):
        """Create corona test tab"""
        self.test_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.test_tab, text="Corona Test")
        
        ttk.Label(self.test_tab, text="Corona Test Assessment", 
                 font=('Arial', 16, 'bold')).grid(row=0, column=0, columnspan=2, pady=10)
        
        # Patient selection
        ttk.Label(self.test_tab, text="Select Patient:").grid(row=1, column=0, sticky='w', padx=5, pady=5)
        self.patient_combo = ttk.Combobox(self.test_tab, state="readonly")
        self.patient_combo.grid(row=1, column=1, padx=5, pady=5)
        
        # Symptoms
        symptoms = [
            ("Temperature (°C)", "temp_entry"),
            ("Cough", "cough_var"),
            ("Fever", "fever_var"),
            ("Breathing Difficulty", "breathing_var"),
            ("Loss of Taste/Smell", "taste_var"),
            ("Fatigue", "fatigue_var"),
            ("Chest Pain", "chest_var")
        ]
        
        self.test_entries = {}
        for i, (label, key) in enumerate(symptoms):
            ttk.Label(self.test_tab, text=label+":").grid(row=i+2, column=0, sticky='w', padx=5, pady=5)
            if key == "temp_entry":
                entry = ttk.Entry(self.test_tab, width=10)
            else:
                entry = ttk.Checkbutton(self.test_tab, text="Yes")
                var = tk.IntVar()
                entry.config(variable=var)
                self.test_entries[key] = var
                entry.grid(row=i+2, column=1, sticky='w', padx=5, pady=5)
                continue
            entry.grid(row=i+2, column=1, sticky='w', padx=5, pady=5)
            self.test_entries[key] = entry
        
        # Risk factors
        ttk.Label(self.test_tab, text="Travel History:").grid(row=9, column=0, sticky='w', padx=5, pady=5)
        self.travel_var = tk.IntVar()
        ttk.Checkbutton(self.test_tab, text="Yes", variable=self.travel_var).grid(row=9, column=1, sticky='w', padx=5, pady=5)
        
        ttk.Label(self.test_tab, text="Contact with Infected:").grid(row=10, column=0, sticky='w', padx=5, pady=5)
        self.contact_var = tk.IntVar()
        ttk.Checkbutton(self.test_tab, text="Yes", variable=self.contact_var).grid(row=10, column=1, sticky='w', padx=5, pady=5)
        
        # Buttons
        button_frame = ttk.Frame(self.test_tab)
        button_frame.grid(row=11, column=0, columnspan=2, pady=10)
        
        ttk.Button(button_frame, text="Conduct Test", 
                  command=self.conduct_test).pack(side=tk.LEFT, padx=5)
        ttk.Button(button_frame, text="Clear Form", 
                  command=self.clear_test_form).pack(side=tk.LEFT, padx=5)
    
    def create_results_tab(self):
        """Create test results tab"""
        self.results_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.results_tab, text="Test Results")
        
        ttk.Label(self.results_tab, text="Test Results", 
                 font=('Arial', 16, 'bold')).pack(pady=10)
        
        # Results treeview
        self.results_tree = ttk.Treeview(self.results_tab, 
                                        columns=("ID", "Patient", "Date", "Result", "Severity"),
                                        show="headings")
        for col in ["ID", "Patient", "Date", "Result", "Severity"]:
            self.results_tree.heading(col, text=col)
            self.results_tree.column(col, width=120)
        self.results_tree.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Result details frame
        details_frame = ttk.LabelFrame(self.results_tab, text="Test Details")
        details_frame.pack(fill=tk.X, padx=10, pady=10)
        
        self.result_text = tk.Text(details_frame, height=8, width=80)
        self.result_text.pack(padx=5, pady=5)
        
        # Bind selection event
        self.results_tree.bind('<<TreeviewSelect>>', self.show_result_details)
    
    def create_analytics_tab(self):
        """Create analytics tab"""
        self.analytics_tab = ttk.Frame(self.notebook)
        self.notebook.add(self.analytics_tab, text="Analytics")
        
        # Statistics frame
        stats_frame = ttk.LabelFrame(self.analytics_tab, text="Statistics")
        stats_frame.pack(fill=tk.X, padx=10, pady=10)
        
        self.stats_text = tk.Text(stats_frame, height=6, width=80)
        self.stats_text.pack(padx=5, pady=5)
        
        # Charts frame
        charts_frame = ttk.LabelFrame(self.analytics_tab, text="Charts")
        charts_frame.pack(fill=tk.BOTH, expand=True, padx=10, pady=10)
        
        # Create matplotlib figure
        self.fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(10, 8))
        self.fig.tight_layout(pad=5.0)
        
        self.canvas = FigureCanvasTkAgg(self.fig, charts_frame)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)
        
        self.update_analytics()
    
    def register_patient(self):
        """Register a new patient"""
        try:
            name = self.patient_entries['name_entry'].get()
            age = self.patient_entries['age_entry'].get()
            gender = self.patient_entries['gender_combo'].get()
            phone = self.patient_entries['phone_entry'].get()
            address = self.patient_entries['address_entry'].get()
            
            if not all([name, age, gender]):
                messagebox.showerror("Error", "Please fill all required fields")
                return
            
            registration_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
            self.cursor.execute('''
                INSERT INTO patients (name, age, gender, phone, address, registration_date)
                VALUES (?, ?, ?, ?, ?, ?)
            ''', (name, age, gender, phone, address, registration_date))
            
            self.conn.commit()
            messagebox.showinfo("Success", "Patient registered successfully!")
            self.clear_patient_form()
            self.load_patients()
            
        except Exception as e:
            messagebox.showerror("Error", f"Failed to register patient: {str(e)}")
    
    def clear_patient_form(self):
        """Clear patient registration form"""
        for entry in self.patient_entries.values():
            if isinstance(entry, ttk.Combobox):
                entry.set('')
            else:
                entry.delete(0, tk.END)
    
    def load_patients(self):
        """Load patients into combobox and treeview"""
        # Clear existing data
        for item in self.patient_tree.get_children():
            self.patient_tree.delete(item)
        
        patients = []
        self.cursor.execute("SELECT id, name, age, gender, phone FROM patients")
        rows = self.cursor.fetchall()
        
        for row in rows:
            self.patient_tree.insert("", tk.END, values=row)
            patients.append(f"{row[0]} - {row[1]}")
        
        self.patient_combo['values'] = patients
    
    def conduct_test(self):
        """Conduct corona test and save results"""
        try:
            patient_selection = self.patient_combo.get()
            if not patient_selection:
                messagebox.showerror("Error", "Please select a patient")
                return
            
            patient_id = patient_selection.split(" - ")[0]
            
            # Get symptoms data
            temperature = float(self.test_entries['temp_entry'].get() or 0)
            cough = self.test_entries['cough_var'].get()
            fever = self.test_entries['fever_var'].get()
            breathing = self.test_entries['breathing_var'].get()
            taste = self.test_entries['taste_var'].get()
            fatigue = self.test_entries['fatigue_var'].get()
            chest = self.test_entries['chest_var'].get()
            
            # Calculate risk score
            risk_score = 0
            symptoms_count = 0
            
            if temperature > 37.5: risk_score += 2
            if cough: risk_score += 1; symptoms_count += 1
            if fever: risk_score += 2; symptoms_count += 1
            if breathing: risk_score += 3; symptoms_count += 1
            if taste: risk_score += 2; symptoms_count += 1
            if fatigue: risk_score += 1; symptoms_count += 1
            if chest: risk_score += 3; symptoms_count += 1
            if self.travel_var.get(): risk_score += 2
            if self.contact_var.get(): risk_score += 3
            
            # Determine result and severity
            if risk_score >= 8:
                result = "Positive"
                if risk_score >= 12:
                    severity = "High"
                else:
                    severity = "Medium"
            else:
                result = "Negative"
                severity = "Low"
            
            # Save test results
            test_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            
            self.cursor.execute('''
                INSERT INTO tests (patient_id, test_date, temperature, cough, fever, 
                                breathing_difficulty, loss_of_taste_smell, fatigue, 
                                chest_pain, travel_history, contact_with_infected,
                                test_result, severity)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (patient_id, test_date, temperature, cough, fever, breathing, 
                  taste, fatigue, chest, self.travel_var.get(), 
                  self.contact_var.get(), result, severity))
            
            self.conn.commit()
            
            # Show result
            message = f"Test Result: {result}\nSeverity: {severity}\nRisk Score: {risk_score}\nSymptoms Count: {symptoms_count}"
            messagebox.showinfo("Test Completed", message)
            
            self.clear_test_form()
            self.load_tests()
            self.update_analytics()
            
        except ValueError:
            messagebox.showerror("Error", "Please enter valid temperature")
        except Exception as e:
            messagebox.showerror("Error", f"Failed to conduct test: {str(e)}")
    
    def clear_test_form(self):
        """Clear test form"""
        self.patient_combo.set('')
        self.test_entries['temp_entry'].delete(0, tk.END)
        for var in self.test_entries.values():
            if isinstance(var, tk.IntVar):
                var.set(0)
        self.travel_var.set(0)
        self.contact_var.set(0)
    
    def load_tests(self):
        """Load test results into treeview"""
        for item in self.results_tree.get_children():
            self.results_tree.delete(item)
        
        self.cursor.execute('''
            SELECT t.id, p.name, t.test_date, t.test_result, t.severity
            FROM tests t
            JOIN patients p ON t.patient_id = p.id
            ORDER BY t.test_date DESC
        ''')
        
        rows = self.cursor.fetchall()
        for row in rows:
            self.results_tree.insert("", tk.END, values=row)
    
    def show_result_details(self, event):
        """Show detailed test results when selected"""
        selection = self.results_tree.selection()
        if not selection:
            return
        
        item = self.results_tree.item(selection[0])
        test_id = item['values'][0]
        
        self.cursor.execute('''
            SELECT t.*, p.name, p.age, p.gender
            FROM tests t
            JOIN patients p ON t.patient_id = p.id
            WHERE t.id = ?
        ''', (test_id,))
        
        test_data = self.cursor.fetchone()
        
        if test_data:
            details = f"""Patient: {test_data[13]} (Age: {test_data[14]}, Gender: {test_data[15]})
Test Date: {test_data[2]}
Temperature: {test_data[3]}°C

Symptoms:
• Cough: {'Yes' if test_data[4] else 'No'}
• Fever: {'Yes' if test_data[5] else 'No'}
• Breathing Difficulty: {'Yes' if test_data[6] else 'No'}
• Loss of Taste/Smell: {'Yes' if test_data[7] else 'No'}
• Fatigue: {'Yes' if test_data[8] else 'No'}
• Chest Pain: {'Yes' if test_data[9] else 'No'}

Risk Factors:
• Travel History: {'Yes' if test_data[10] else 'No'}
• Contact with Infected: {'Yes' if test_data[11] else 'No'}

Result: {test_data[12]}
Severity: {test_data[13]}"""
            
            self.result_text.delete(1.0, tk.END)
            self.result_text.insert(1.0, details)
    
    def update_analytics(self):
        """Update analytics and charts"""
        # Get statistics
        self.cursor.execute("SELECT COUNT(*) FROM tests")
        total_tests = self.cursor.fetchone()[0]
        
        self.cursor.execute("SELECT COUNT(*) FROM tests WHERE test_result = 'Positive'")
        positive_tests = self.cursor.fetchone()[0]
        
        self.cursor.execute("SELECT COUNT(*) FROM tests WHERE test_result = 'Negative'")
        negative_tests = self.cursor.fetchone()[0]
        
        self.cursor.execute('''
            SELECT severity, COUNT(*) FROM tests 
            WHERE test_result = 'Positive' GROUP BY severity
        ''')
        severity_data = self.cursor.fetchall()
        
        # Update statistics text
        stats_text = f"""Total Tests: {total_tests}
Positive Cases: {positive_tests}
Negative Cases: {negative_tests}
Positive Rate: {positive_tests/total_tests*100:.1f}% if total_tests > 0 else 0%

Severity Distribution:"""
        for severity, count in severity_data:
            stats_text += f"\n• {severity}: {count} cases"
        
        self.stats_text.delete(1.0, tk.END)
        self.stats_text.insert(1.0, stats_text)
        
        # Update charts
        self.update_charts(positive_tests, negative_tests, severity_data)
    
    def update_charts(self, positive, negative, severity_data):
        """Update the analytics charts"""
        # Clear previous plots
        for ax in self.fig.axes:
            ax.clear()
        
        ax1, ax2, ax3, ax4 = self.fig.axes
        
        # Chart 1: Test Results Pie Chart
        if positive + negative > 0:
            labels = ['Positive', 'Negative']
            sizes = [positive, negative]
            colors = ['#ff6b6b', '#51cf66']
            ax1.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
            ax1.set_title('Test Results Distribution')
        
        # Chart 2: Severity Distribution
        if severity_data:
            severities = [item[0] for item in severity_data]
            counts = [item[1] for item in severity_data]
            colors = ['#ffa8a8', '#ff6b6b', '#c92a2a']
            ax2.bar(severities, counts, color=colors[:len(severities)])
            ax2.set_title('Positive Cases by Severity')
            ax2.set_ylabel('Number of Cases')
        
        # Chart 3: Symptoms Analysis
        self.cursor.execute('''
            SELECT 
                SUM(cough), SUM(fever), SUM(breathing_difficulty),
                SUM(loss_of_taste_smell), SUM(fatigue), SUM(chest_pain)
            FROM tests WHERE test_result = 'Positive'
        ''')
        symptom_data = self.cursor.fetchone()
        
        if symptom_data and any(symptom_data):
            symptoms = ['Cough', 'Fever', 'Breathing', 'Taste/Smell', 'Fatigue', 'Chest Pain']
            counts = list(symptom_data)
            ax3.bar(symptoms, counts, color='#74c0fc')
            ax3.set_title('Common Symptoms in Positive Cases')
            ax3.set_ylabel('Count')
            ax3.tick_params(axis='x', rotation=45)
        
        # Chart 4: Daily Tests (last 7 days)
        self.cursor.execute('''
            SELECT DATE(test_date), COUNT(*), 
                   SUM(CASE WHEN test_result = 'Positive' THEN 1 ELSE 0 END)
            FROM tests 
            WHERE test_date >= date('now', '-7 days')
            GROUP BY DATE(test_date)
            ORDER BY DATE(test_date)
        ''')
        daily_data = self.cursor.fetchall()
        
        if daily_data:
            dates = [item[0] for item in daily_data]
            total = [item[1] for item in daily_data]
            positive = [item[2] for item in daily_data]
            
            ax4.plot(dates, total, marker='o', label='Total Tests', color='#339af0')
            ax4.plot(dates, positive, marker='s', label='Positive Cases', color='#ff6b6b')
            ax4.set_title('Tests (Last 7 Days)')
            ax4.set_ylabel('Number of Tests')
            ax4.legend()
            ax4.tick_params(axis='x', rotation=45)
        
        self.canvas.draw()
    
    def __del__(self):
        """Close database connection"""
        if hasattr(self, 'conn'):
            self.conn.close()

def main():
    root = tk.Tk()
    app = CoronaDetectionSystem(root)
    root.mainloop()

if __name__ == "__main__":
    main()